In [ ]:
!git clone https://github.com/VuTrinhNguyenHoang/incremental-blood-cell-classification.git
%cd incremental-blood-cell-classification
%pip install -q -e .

In [3]:
import json
from pathlib import Path

import torch

from incremental_blood_cell.config import ExperimentConfig
from incremental_blood_cell.data import load_bloodmnist
from incremental_blood_cell.experiment import run_experiment

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4


In [4]:
data_root = Path("/kaggle/working/data")

train_dataset = load_bloodmnist(
    split="train",
    root=data_root,
    download=True,
)

test_dataset = load_bloodmnist(
    split="test",
    root=data_root,
    download=True,
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

100%|██████████| 156M/156M [00:03<00:00, 39.4MB/s] 


Train samples: 11959
Test samples: 3421


In [5]:
class_order = (0, 1, 2, 3, 4, 5, 6, 7)

configs = [
    ExperimentConfig(
        method="finetuning",
        class_order=class_order,
        seed=0,
        epochs=1,
        batch_size=128,
        learning_rate=1e-3,
    ),
    ExperimentConfig(
        method="hybrid",
        class_order=class_order,
        seed=0,
        epochs=1,
        batch_size=128,
        learning_rate=1e-3,
        memory_size=40,
        distillation_weight=1.0,
        temperature=2.0,
    ),
]

In [6]:
results = {}

for config in configs:
    print(f"\nRunning: {config.method}")

    result = run_experiment(
        config=config,
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        device=device,
        show_progress=True,
    )

    result.model.to("cpu")
    results[config.method] = result

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Running: finetuning
Experience 1/3 | classes=(0, 1, 2, 3)


Training:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Experience 1/3 | loss=0.2748 | avg_acc=0.6832
Experience 2/3 | classes=(4, 5)


Training:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Experience 2/3 | loss=2.2159 | avg_acc=0.2328 | memory=40 | forgetting=0.6769
Experience 3/3 | classes=(6, 7) | selection=hybrid


Training:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Experience 3/3 | loss=1.3750 | avg_acc=0.3292 | memory=40 | forgetting=0.5713


In [7]:
for method, result in results.items():
    assert tuple(map(len, result.accuracy_matrix)) == (1, 2, 3)
    assert result.model.fc.out_features == 8

    for row in result.accuracy_matrix:
        assert all(0.0 <= accuracy <= 1.0 for accuracy in row)

    print(
        method,
        {
            "final_average_accuracy": result.final_average_accuracy,
            "average_forgetting": result.average_forgetting,
            "backward_transfer": result.backward_transfer,
        },
    )

print("Smoke test passed.")

finetuning {'final_average_accuracy': 0.33186619718309857, 'average_forgetting': 0.5721316270645658, 'backward_transfer': -0.5721316270645658}
hybrid {'final_average_accuracy': 0.32922535211267606, 'average_forgetting': 0.5712783847437467, 'backward_transfer': -0.5712783847437467}
Smoke test passed.


In [8]:
output_dir = Path("/kaggle/working/outputs/smoke")
output_dir.mkdir(parents=True, exist_ok=True)

summaries = {
    method: result.to_dict()
    for method, result in results.items()
}

with (output_dir / "results.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(summaries, file, indent=2)

for method, result in results.items():
    torch.save(
        {
            "config": result.to_dict()["config"],
            "model_state_dict": result.model.state_dict(),
        },
        output_dir / f"{method}.pt",
    )

print(*sorted(output_dir.iterdir()), sep="\n")

/kaggle/working/outputs/smoke/finetuning.pt
/kaggle/working/outputs/smoke/hybrid.pt
/kaggle/working/outputs/smoke/results.json
